# Vault Enterprise en Minikube

La demo completa se inicia desde la raíz con **`./demo.sh`**. Este notebook documenta solo la etapa de Vault y usa los mismos scripts, sin borrar perfiles de Minikube ni reinicializar un cluster existente.

[Instalación y secretos de entrada](README.md) · [Revisión del workspace](docs/WORKSPACE_REVIEW.md).

Preparar `.env` a partir de `.env.example`: licencia, credenciales externas de integración y configuración de la imagen. No añadir `VAULT_TOKEN`, `VAULT_ADDR` ni `VAULT_CACERT`: el lanzador genera el bootstrap después de inicializar Vault. El estado generado (CA, claves de inicialización, token administrativo, logs y PID) se guarda en `~/.local/state/lumen-vault-demo`, fuera de `/tmp` y del workspace. `LUMEN_STATE_DIR` permite otra ubicación. Para adoptar un Vault ya inicializado se necesitan sus claves guardadas; no se regeneran.


In [ ]:
%%bash
set -euo pipefail
./demo.sh check


## Despliegue reejecutable

Crea o arranca el perfil `workshop`; conserva los recursos existentes. En una instalación vacía descarga de Docker Hub la imagen `josemerchan/vault-xai:2.1.2-ent` con el plugin xAI incluido (AMD64/ARM64), despliega tres nodos Raft con TLS, inicializa una sola vez y desbloquea los nodos. `.env` permite configurar `VAULT_IMAGE_REPOSITORY` y `VAULT_IMAGE_TAG`. El puerto 8443 escucha en `127.0.0.1`.

Si ya existe Vault, comprueba la imagen y adopta las claves del laboratorio anterior; no reemplaza la imagen ni regenera certificados. El catálogo del plugin es global, pero el motor xAI utilizado por Lumen se configura en `verify`.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/vault.sh


In [ ]:
%%bash
source scripts/demo/common.sh
load_inputs
load_vault
vault status
kubectl --context workshop -n vault get pods
helm --kube-context workshop -n vault list


Para reanudar, usar `./demo.sh start`; no ejecutar `minikube delete --all`, borrar claves manualmente ni volver a hacer `operator init`. `./demo.sh stop` detiene solo procesos locales, conservando cluster y datos. Para retirar el laboratorio completo, revisar `./demo.sh remove --dry-run` y ejecutar `./demo.sh remove`: elimina los tres namespaces/PVC y los recursos Lumen de Verify, conserva `.env` y permite un nuevo `start`. [Alcance de la eliminación](README.md#volver-a-empezar-desde-cero). La inicialización de un solo fragmento es una simplificación del laboratorio.
